# SMOTE for Images - Decoder Architecture Comparison

This notebook compares different decoder architectures available in the SMOTE Image Synthesis pipeline.

In [ ]:
import torch
import numpy as np
import sys
import os

# Add the project root to Python path
sys.path.insert(0, os.path.dirname(os.getcwd()))

# Import pipeline components
from smote_image_synthesis.data.models import PipelineConfig
from smote_image_synthesis.encoders.resnet_encoder import ResNetEncoder
from smote_image_synthesis.decoders.autoencoder_decoder import AutoencoderDecoder
from smote_image_synthesis.decoders.vae_decoder import VAEDecoder
from smote_image_synthesis.decoders.gan_decoder import GANDecoder
from smote_image_synthesis.smote.constrained_smote import ConstrainedSMOTE
from smote_image_synthesis.quality.assessor import QualityAssessor
from smote_image_synthesis.pipeline import SynthesisPipeline

## Create Sample Data

Generate a small synthetic dataset to demonstrate the pipeline.

In [ ]:
# Create synthetic dataset
def create_synthetic_dataset(n_samples=80, image_size=64):
    """Create a synthetic dataset for demonstration."""
    images = []
    labels = []
    
    # Create imbalanced distribution
    class_sizes = [int(n_samples * 0.5), int(n_samples * 0.3), int(n_samples * 0.2)]
    
    # Class 0: Horizontal stripes (majority)
    for i in range(class_sizes[0]):
        img = torch.zeros(3, image_size, image_size)
        stripe_width = 8
        for y in range(0, image_size, stripe_width * 2):
            img[:, y:y+stripe_width, :] = torch.rand(3, 1, 1) * 0.8 + 0.2
        
        # Add noise
        img += torch.randn_like(img) * 0.1
        img = torch.clamp(img, 0, 1)
        
        images.append(img)
        labels.append(0)
    
    # Class 1: Vertical stripes (minority)
    for i in range(class_sizes[1]):
        img = torch.zeros(3, image_size, image_size)
        stripe_width = 8
        for x in range(0, image_size, stripe_width * 2):
            img[:, :, x:x+stripe_width] = torch.rand(3, 1, 1) * 0.8 + 0.2
        
        # Add noise
        img += torch.randn_like(img) * 0.1
        img = torch.clamp(img, 0, 1)
        
        images.append(img)
        labels.append(1)
    
    # Class 2: Checkerboard pattern (minority)
    for i in range(class_sizes[2]):
        img = torch.zeros(3, image_size, image_size)
        square_size = 16
        
        for y in range(0, image_size, square_size):
            for x in range(0, image_size, square_size):
                if (y // square_size + x // square_size) % 2 == 0:
                    color = torch.rand(3) * 0.8 + 0.2
                    img[:, y:y+square_size, x:x+square_size] = color.view(3, 1, 1)
        
        # Add noise
        img += torch.randn_like(img) * 0.1
        img = torch.clamp(img, 0, 1)
        
        images.append(img)
        labels.append(2)
    
    images_tensor = torch.stack(images)
    labels_array = np.array(labels)
    
    print(f"Dataset created with shape {images_tensor.shape}")
    print(f"Class distribution: {np.bincount(labels_array)}")
    
    return images_tensor, labels_array

# Create dataset
images, labels = create_synthetic_dataset(n_samples=60, image_size=64)

## Define Common Configuration

Set up a common configuration that will be used across all decoder types.

In [ ]:
# Common configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Create base configuration
base_config = {
    'embedding_dim': 128,
    'image_shape': (3, 64, 64),
    'device': device
}

# Create shared encoder and SMOTE
encoder = ResNetEncoder(
    architecture='resnet18',
    embedding_dim=base_config['embedding_dim'],
    pretrained=False,
    device=device
)

smote = ConstrainedSMOTE(
    k_neighbors=3,
    use_clustering=True,
    clustering_method='kmeans',
    random_state=42
)

quality_assessor = QualityAssessor(
    metrics=['mse', 'mae'],
    device=device
)

## Autoencoder Decoder

Test the autoencoder decoder architecture.

In [ ]:
print("Testing Autoencoder Decoder...")

# Create autoencoder decoder
ae_decoder = AutoencoderDecoder(
    embedding_dim=base_config['embedding_dim'],
    image_shape=base_config['image_shape'],
    device=device
)

# Create pipeline with autoencoder
ae_pipeline = SynthesisPipeline(
    encoder=encoder,
    decoder=ae_decoder,
    smote=smote,
    quality_assessor=quality_assessor
)

# Fit pipeline
ae_pipeline.fit(images, labels, train_decoder=False)

# Generate synthetic images
ae_synthetic_images, ae_synthetic_labels = ae_pipeline.generate_synthetic_images(20)

print(f"Autoencoder: Generated {len(ae_synthetic_images)} synthetic images")

# Evaluate quality
if len(ae_synthetic_images) > 0:
    n_eval = min(10, len(images), len(ae_synthetic_images))
    eval_real = images[:n_eval]
    eval_synthetic = ae_synthetic_images[:n_eval]
    
    ae_quality_results = ae_pipeline.evaluate_quality(eval_synthetic, eval_real)
    print("Autoencoder Quality Results:")
    for metric, value in ae_quality_results.items():
        print(f"  {metric}: {value:.6f}")
else:
    print("Autoencoder: No synthetic images generated")

## VAE Decoder

Test the VAE decoder architecture.

In [ ]:
print("\nTesting VAE Decoder...")

# Create VAE decoder
vae_decoder = VAEDecoder(
    embedding_dim=base_config['embedding_dim'],
    image_shape=base_config['image_shape'],
    latent_dim=64,  # Smaller latent dimension for demo
    device=device
)

# Create pipeline with VAE
vae_pipeline = SynthesisPipeline(
    encoder=encoder,
    decoder=vae_decoder,
    smote=smote,
    quality_assessor=quality_assessor
)

# Fit pipeline
vae_pipeline.fit(images, labels, train_decoder=False)

# Generate synthetic images
vae_synthetic_images, vae_synthetic_labels = vae_pipeline.generate_synthetic_images(20)

print(f"VAE: Generated {len(vae_synthetic_images)} synthetic images")

# Evaluate quality
if len(vae_synthetic_images) > 0:
    n_eval = min(10, len(images), len(vae_synthetic_images))
    eval_real = images[:n_eval]
    eval_synthetic = vae_synthetic_images[:n_eval]
    
    vae_quality_results = vae_pipeline.evaluate_quality(eval_synthetic, eval_real)
    print("VAE Quality Results:")
    for metric, value in vae_quality_results.items():
        print(f"  {metric}: {value:.6f}")
else:
    print("VAE: No synthetic images generated")

## GAN Decoder

Test the GAN decoder architecture.

In [ ]:
print("\nTesting GAN Decoder...")

# Create GAN decoder
gan_decoder = GANDecoder(
    embedding_dim=base_config['embedding_dim'],
    image_shape=base_config['image_shape'],
    latent_dim=64,  # Smaller latent dimension for demo
    device=device
)

# Create pipeline with GAN
gan_pipeline = SynthesisPipeline(
    encoder=encoder,
    decoder=gan_decoder,
    smote=smote,
    quality_assessor=quality_assessor
)

# Fit pipeline
gan_pipeline.fit(images, labels, train_decoder=False)

# Generate synthetic images
gan_synthetic_images, gan_synthetic_labels = gan_pipeline.generate_synthetic_images(20)

print(f"GAN: Generated {len(gan_synthetic_images)} synthetic images")

# Evaluate quality
if len(gan_synthetic_images) > 0:
    n_eval = min(10, len(images), len(gan_synthetic_images))
    eval_real = images[:n_eval]
    eval_synthetic = gan_synthetic_images[:n_eval]
    
    gan_quality_results = gan_pipeline.evaluate_quality(eval_synthetic, eval_real)
    print("GAN Quality Results:")
    for metric, value in gan_quality_results.items():
        print(f"  {metric}: {value:.6f}")
else:
    print("GAN: No synthetic images generated")

## Comparison Summary

Compare the results from different decoder architectures.

In [ ]:
print("\n=== DECODER ARCHITECTURE COMPARISON ===")
print(f"Autoencoder: {len(ae_synthetic_images)} images generated")
if len(ae_synthetic_images) > 0 and 'ae_quality_results' in locals():
    for metric, value in ae_quality_results.items():
        print(f"  AE {metric}: {value:.6f}")

print(f"VAE: {len(vae_synthetic_images)} images generated")
if len(vae_synthetic_images) > 0 and 'vae_quality_results' in locals():
    for metric, value in vae_quality_results.items():
        print(f"  VAE {metric}: {value:.6f}")

print(f"GAN: {len(gan_synthetic_images)} images generated")
if len(gan_synthetic_images) > 0 and 'gan_quality_results' in locals():
    for metric, value in gan_quality_results.items():
        print(f"  GAN {metric}: {value:.6f}")

print("\nDecoder Architecture Characteristics:")
print("- Autoencoder: Good reconstruction quality, deterministic")
print("- VAE: Probabilistic, allows for sampling variations")
print("- GAN: Potentially highest quality, but harder to train")

## When to Use Each Architecture

### Autoencoder
- Best for: Reconstruction-focused tasks
- Advantages: Stable training, deterministic outputs
- Disadvantages: May produce blurry images

### VAE (Variational Autoencoder)
- Best for: When you need diverse samples from the same class
- Advantages: Probabilistic, allows for sampling variations
- Disadvantages: May produce less sharp images

### GAN (Generative Adversarial Network)
- Best for: Highest quality synthetic images
- Advantages: Can produce very realistic images
- Disadvantages: Harder to train, may suffer from mode collapse

Choose the architecture based on your specific requirements for image quality, diversity, and training stability.